# DSA Week 13 -- Final Integration

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Focus:** Wire all DSA optimizations into your product pipeline

## Learning Objectives

1. Integrate your optimized DSA module into the main pipeline
2. Implement a config switch between baseline and optimized modes
3. Run the complete pipeline end-to-end
4. Verify all tests still pass with the integrated optimization
5. Prepare for final demo and benchmark report

## Integration Checklist

```
+--------------------------------------------------+
|           FINAL INTEGRATION CHECKLIST             |
+--------------------------------------------------+
| [ ] Optimized module in src/<project>/dsa/        |
| [ ] Module imports cleanly into pipeline          |
| [ ] Config switch: baseline vs optimized          |
| [ ] All existing tests pass                       |
| [ ] New tests cover the optimized module          |
| [ ] Benchmark shows >= 1.5x at largest n          |
| [ ] benchmark_results.json generated              |
| [ ] benchmark_plot.png generated                  |
| [ ] Complexity note in report                     |
+--------------------------------------------------+
```

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Integration Architecture

Your pipeline should support switching between baseline and optimized modes:

```python
config = {
    "mode": "optimized",  # or "baseline"
    # ... other settings
}

if config["mode"] == "optimized":
    from project.dsa import optimized_lookup as lookup
else:
    from project.dsa import baseline_lookup as lookup
```

In [ ]:
# Example integration pattern
class Pipeline:
    """Example pipeline with configurable optimization."""

    def __init__(self, config):
        self.config = config
        self.mode = config.get("mode", "baseline")
        print("Pipeline initialized in " + self.mode + " mode")

    def find_records(self, data, target_key, target_value):
        """Find records matching criteria -- uses configured mode."""
        if self.mode == "optimized":
            return self._find_records_optimized(data, target_key, target_value)
        else:
            return self._find_records_baseline(data, target_key, target_value)

    def _find_records_baseline(self, data, key, value):
        """Baseline: linear scan. O(n)."""
        return [r for r in data if r.get(key) == value]

    def _find_records_optimized(self, data, key, value):
        """Optimized: build index if not cached, then O(1) lookup."""
        if not hasattr(self, '_indexes'):
            self._indexes = {}
        if key not in self._indexes:
            # Build index once
            idx = {}
            for i, r in enumerate(data):
                k = r.get(key)
                if k not in idx:
                    idx[k] = []
                idx[k].append(i)
            self._indexes[key] = idx
        # O(1) lookup
        indices = self._indexes[key].get(value, [])
        return [data[i] for i in indices]

# Demo
import random
random.seed(42)
data = [{"id": i, "category": "cat_" + str(i % 20), "value": random.random()} for i in range(50_000)]

# Test both modes give same result
config_base = {"mode": "baseline"}
config_opt = {"mode": "optimized"}

pipe_base = Pipeline(config_base)
pipe_opt = Pipeline(config_opt)

r1 = pipe_base.find_records(data, "category", "cat_5")
r2 = pipe_opt.find_records(data, "category", "cat_5")

print()
print("Baseline found: " + str(len(r1)) + " records")
print("Optimized found: " + str(len(r2)) + " records")
print("Results match: " + str(len(r1) == len(r2)))

# Timing
import timeit
t_base = timeit.timeit(lambda: pipe_base.find_records(data, "category", "cat_5"), number=100)
t_opt = timeit.timeit(lambda: pipe_opt.find_records(data, "category", "cat_5"), number=100)
print()
print("Baseline (100 queries): " + "{:.4f}".format(t_base) + "s")
print("Optimized (100 queries): " + "{:.4f}".format(t_opt) + "s")
print("Speedup: " + "{:.0f}".format(t_base / t_opt) + "x")

---
## Part 2: Generate Final Artifacts

In [ ]:
import json
import os

# Generate final benchmark results
results = {
    "description": "Pipeline query optimization",
    "baseline_complexity": "O(n) per query -- linear scan",
    "optimized_complexity": "O(1) per query after O(n) index build",
    "data_structure": "Hash-based index (dict of lists)",
    "sizes": [1000, 5000, 10000, 50000],
    "baseline_ms": [],
    "optimized_ms": [],
    "speedup": [],
}

for n in results["sizes"]:
    test_data = [{"id": i, "category": "cat_" + str(i % 20)} for i in range(n)]
    pb = Pipeline({"mode": "baseline"})
    po = Pipeline({"mode": "optimized"})

    runs = 100
    t_b = timeit.timeit(lambda d=test_data: pb.find_records(d, "category", "cat_5"), number=runs) / runs * 1000
    t_o = timeit.timeit(lambda d=test_data: po.find_records(d, "category", "cat_5"), number=runs) / runs * 1000
    sp = t_b / t_o if t_o > 0 else 0

    results["baseline_ms"].append(round(t_b, 4))
    results["optimized_ms"].append(round(t_o, 4))
    results["speedup"].append(round(sp, 1))

os.makedirs("reports/benchmark", exist_ok=True)
with open("reports/benchmark/benchmark_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved: reports/benchmark/benchmark_results.json")

# Verify target
max_sp = max(results["speedup"])
print("Max speedup: " + str(max_sp) + "x (" + ("MEETS" if max_sp >= 1.5 else "DOES NOT MEET") + " 1.5x target)")

---
## Part 3: Your Integration TODO

Replace the demo code above with your actual project integration.
Make sure:
1. Your optimized module is in `src/<project>/dsa/`
2. Pipeline can switch between modes via config
3. All tests pass in both modes
4. Benchmark artifacts are generated

In [ ]:
# TODO: Your integration code here
print("Replace with your project integration!")

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)